# Ad and adstock spend test (inference only, no retraining)

**Question:** how much does the model's Oct-Dec forecast depend on the two spend columns, and is the planned spend inside the range the model saw in training?

**Part A (no model):** compare the Oct-Dec spend with the training range, path by path.
**Part B (model):** forecast the same sample of series several times, changing only the spend in the forecast window. Petrol is held at a sensible base (default: last year's petrol on the same festival days) so it does not swamp the spend effect.

Run from the same folder as the prediction notebook (`column_roles.json`, `run_metadata.json`, `series_cache/`, the checkpoint).

How to read it: spend is expected to move sales a little, so a response alone is not a bug. The warning signs are the wrong direction (more spend, fewer sales), or a very large swing for a small change.

In [ ]:
import os, json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from darts import TimeSeries
from darts.models import TFTModel
from pytorch_lightning import Callback

BASE      = os.getcwd()
CACHE_DIR = os.path.join(BASE, "series_cache")
OUT_DIR   = os.path.join(BASE, "spend_override_test")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- settings ----------------
N_SAMPLE    = 2000          # series in the test; None = all (slow). Same series in every scenario.
SEED        = 0
BATCH_SIZE  = 512
PETROL_COL  = None          # None = find the one shared column with "PETROL" in its name
PETROL_BASE = "last_year"   # petrol used in every spend scenario: "last_year" | "as_is" | a z-score number such as 0.0
SD_SHIFT    = 1.0           # the "plus/minus" scenarios move spend by this many z-score units

In [ ]:
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
with open(os.path.join(BASE, "run_metadata.json")) as f:
    META = json.load(f)

time_col, target_col, FREQ = ROLES["time_col"], ROLES["target_col"], ROLES["freq"]
static_covariates = ROLES["static_covariates"]
series_covariates = ROLES["series_covariates"]
future_covariates = ROLES["future_covariates"]
INPUT_CHUNK_LENGTH = ROLES["input_chunk_length"]
FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END   = pd.Timestamp(ROLES["forecast_end"])
TRAIN_END      = pd.Timestamp(ROLES["train_end"])
HORIZON = (FORECAST_END - FORECAST_START).days + 1
FC_DAYS = pd.date_range(FORECAST_START, FORECAST_END, freq=FREQ)
MODEL_NAME = META["model_name"]
NCOL = len(series_covariates)
safe_name = lambda k: str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

# ---------------- calendar, spend paths ----------------
SHARED_COV  = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
SPEND_PATHS = np.load(os.path.join(CACHE_DIR, "spend_paths.npy"))          # (paths, days, spend columns)
CAL_INDEX   = SHARED_COV.time_index
CAL_VALUES  = SHARED_COV.values(copy=True)
shared_cols = list(SHARED_COV.components)
COV_COLUMNS = shared_cols + list(series_covariates)
assert COV_COLUMNS == future_covariates, "covariate order differs from column_roles.json"
assert SPEND_PATHS.shape[1] == len(CAL_INDEX) and SPEND_PATHS.shape[2] == NCOL
assert CAL_INDEX[0] <= FORECAST_START - pd.Timedelta(days=INPUT_CHUNK_LENGTH) and CAL_INDEX[-1] >= FORECAST_END
U = len(SPEND_PATHS)

fi0, fi1 = CAL_INDEX.get_loc(FORECAST_START), CAL_INDEX.get_loc(FORECAST_END)
ti1 = CAL_INDEX.get_loc(TRAIN_END)

# ---------------- Diwali alignment with last year ----------------
d_flag = pd.Series(CAL_VALUES[:, shared_cols.index("D")], index=CAL_INDEX)
def diwali(year):
    days = d_flag[(d_flag.index.year == year) & (d_flag.index.month >= 9) & (d_flag != 0)].index
    assert len(days) == 1, f"expected one D day in {year}, found {len(days)}"
    return days[0]
D_NOW, D_LY = diwali(FORECAST_START.year), diwali(FORECAST_START.year - 1)
SHIFT = D_NOW - D_LY
LY_DAYS = FC_DAYS - SHIFT
LY_IDX  = np.array([CAL_INDEX.get_loc(d) for d in LY_DAYS])
dwin = (FC_DAYS >= D_NOW - pd.Timedelta(days=3)) & (FC_DAYS <= D_NOW + pd.Timedelta(days=6))
print(f"Spend columns: {list(series_covariates)} | {U} spend paths")
print(f"Diwali {D_LY.date()} (last year) vs {D_NOW.date()} (this year): shift {SHIFT.days} days")

## Part A: is the planned spend inside the range the model saw in training?  (no model needed)

In [ ]:
rows = []
for u in range(U):
    for j, c in enumerate(series_covariates):
        tr = SPEND_PATHS[u, :ti1 + 1, j]
        fc = SPEND_PATHS[u, fi0:fi1 + 1, j]
        ly = SPEND_PATHS[u, LY_IDX, j]
        out_hi, out_lo = (fc > tr.max()), (fc < tr.min())
        rows.append({
            "path": u, "column": c,
            "train min": tr.min(), "train max": tr.max(),
            "plan min": fc.min(), "plan max": fc.max(),
            "% plan days outside train range": 100 * (out_hi | out_lo).mean(),
            "plan mean in Diwali window": fc[dwin].mean(),
            "last-year mean, same days": ly[dwin].mean(),
            "plan flat?": bool(np.ptp(fc) < 1e-4),
        })
rngchk = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print(rngchk.to_string(index=False))
bad = rngchk[rngchk["% plan days outside train range"] > 0]
print(f"\n{len(bad)} of {len(rngchk)} path-columns have planned days outside the training range; "
      f"{int(rngchk['plan flat?'].sum())} are flat (flat would mean carried forward).")
rngchk.to_csv(os.path.join(OUT_DIR, "spend_range_check.csv"), index=False)

## Part B: forecast with changed spend

In [ ]:
# ---------------- model ----------------
try:
    import tft_checks  # noqa: F401   (lets pickled callbacks resolve)
except ImportError:
    print("tft_checks not importable; ok unless loading fails on EmbeddingGate")
class TBDiagnostics(Callback):
    def __init__(self, *args, **kwargs):
        pass

model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)
assert 0 < HORIZON <= model.output_chunk_length, "horizon must be <= output_chunk_length for this test"
print(f"Loaded best checkpoint {MODEL_NAME}")

# ---------------- series sample, history, last-year actuals ----------------
series_index = pd.read_parquet(os.path.join(CACHE_DIR, "series_index.parquet"))
sel = series_index[series_index["in_forecast"]]
if N_SAMPLE and N_SAMPLE < len(sel):
    sel = sel.sample(n=N_SAMPLE, random_state=SEED)
keys     = sel["key"].tolist()
statics  = [pd.DataFrame([row], columns=static_covariates, dtype=np.float32) for row in sel[static_covariates].to_numpy()]
path_ids = sel["spend_path"].to_numpy()

HIST, LY_ACTUAL = [], np.zeros(HORIZON)
for i, k in enumerate(keys):
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        sales, start = z["val_sales"], str(z["val_start"])
    times = pd.date_range(start, periods=len(sales), freq=FREQ)
    assert times[-1] == FORECAST_START - pd.Timedelta(days=1) and len(sales) >= INPUT_CHUNK_LENGTH
    HIST.append(TimeSeries.from_times_and_values(times, sales.reshape(-1, 1).astype(np.float32),
                                                 columns=[target_col], static_covariates=statics[i]))
    LY_ACTUAL += pd.Series(sales, index=times).reindex(LY_DAYS).fillna(0).to_numpy()
print(f"Series in the test: {len(keys):,} | last-year actual on the aligned window: {LY_ACTUAL.sum():,.0f}")

# ---------------- petrol base ----------------
if PETROL_COL is None:
    cands = [c for c in shared_cols if "PETROL" in c.upper()]
    assert len(cands) == 1, f"expected exactly one petrol column, found {cands}; set PETROL_COL by hand"
    PETROL_COL = cands[0]
pi = shared_cols.index(PETROL_COL)
Z_AS_IS = CAL_VALUES[fi0:fi1 + 1, pi].copy()
Z_LY    = CAL_VALUES[LY_IDX, pi].copy()
if PETROL_BASE == "last_year":
    Z_BASE = Z_LY
elif PETROL_BASE == "as_is":
    Z_BASE = Z_AS_IS
else:
    Z_BASE = np.full(HORIZON, float(PETROL_BASE), dtype=np.float32)
print(f"Petrol base for the spend scenarios: {PETROL_BASE} (z from {Z_BASE.min():.2f} to {Z_BASE.max():.2f})")

In [ ]:
# ---------------- spend scenarios ----------------
W = slice(fi0, fi1 + 1)
TR_MEAN = SPEND_PATHS[:, :ti1 + 1, :].mean(axis=1)        # each path's own training average, (paths, columns)
TR_MIN  = SPEND_PATHS[:, :ti1 + 1, :].min(axis=1)

def with_const(arr_UC, cols):
    sp = SPEND_PATHS.copy()
    for j in cols:
        sp[:, W, j] = arr_UC[:, j][:, None]
    return sp

def with_shift(dz, cols):
    sp = SPEND_PATHS.copy()
    for j in cols:
        sp[:, W, j] = sp[:, W, j] + dz
    return sp

def with_last_year(cols):
    sp = SPEND_PATHS.copy()
    for j in cols:
        sp[:, W, j] = SPEND_PATHS[:, LY_IDX, j]
    return sp

ALL = list(range(NCOL))
SCEN = [("original run (petrol as is, spend as planned)", Z_AS_IS, SPEND_PATHS),
        ("BASE (petrol base, spend as planned)",           Z_BASE,  SPEND_PATHS)]
SCEN.append(("spend: all columns at own training average", Z_BASE, with_const(TR_MEAN, ALL)))
if NCOL > 1:
    for j in ALL:
        SCEN.append((f"spend: only {series_covariates[j]} at own training average", Z_BASE, with_const(TR_MEAN, [j])))
SCEN += [("spend: last year's values on the same days",  Z_BASE, with_last_year(ALL)),
         (f"spend: plan {SD_SHIFT:+.1f} z",               Z_BASE, with_shift(+SD_SHIFT, ALL)),
         (f"spend: plan {-SD_SHIFT:+.1f} z",              Z_BASE, with_shift(-SD_SHIFT, ALL)),
         ("spend: at each path's training minimum (about zero spend)", Z_BASE, with_const(TR_MIN, ALL))]

def build_cov(petrol_z, spend):
    vals = CAL_VALUES.copy()
    vals[fi0:fi1 + 1, pi] = petrol_z
    cache, out = {}, []
    for pid in path_ids:
        pid = int(pid)
        if pid not in cache:
            v = np.concatenate([vals, spend[pid]], axis=1).astype(np.float32)
            cache[pid] = TimeSeries.from_times_and_values(CAL_INDEX, v, columns=COV_COLUMNS)
        out.append(cache[pid])
    return out

EXPECTED = [f"{target_col}_r", f"{target_col}_p"]
def forecast_mean(petrol_z, spend):
    preds = model.predict(n=HORIZON, series=HIST, future_covariates=build_cov(petrol_z, spend),
                          predict_likelihood_parameters=True, num_samples=1, batch_size=BATCH_SIZE, verbose=False)
    mean = np.zeros((len(preds), HORIZON))
    for i, p in enumerate(preds):
        assert list(p.components) == EXPECTED, list(p.components)
        v = p.values(copy=False).astype(np.float64)
        r, pr = v[:, 0], v[:, 1]
        assert np.isfinite(v).all() and (r > 0).all() and ((pr > 0) & (pr < 1)).all()
        mean[i] = r * pr / (1.0 - pr)
    return mean

DAILY, t0 = {}, datetime.now()
for name, zp, sp in SCEN:
    DAILY[name] = forecast_mean(zp, sp).sum(axis=0)
    print(f"{name:62s} total {DAILY[name].sum():>12,.0f}   ({(datetime.now() - t0).seconds}s)")

In [ ]:
# ---------------- report ----------------
BASE_NAME = "BASE (petrol base, spend as planned)"
ly_dwin = float(LY_ACTUAL[dwin].sum())
b_tot, b_dw = DAILY[BASE_NAME].sum(), DAILY[BASE_NAME][dwin].sum()
rows = []
for name, d in DAILY.items():
    s = pd.Series(d, index=FC_DAYS)
    r = {"scenario": name, "total": s.sum(), "Diwali window": s[dwin].sum(),
         "total vs BASE": s.sum() / b_tot, "Diwali window vs BASE": s[dwin].sum() / b_dw,
         "vs last-year (all days)": s.sum() / LY_ACTUAL.sum(), "vs last-year (Diwali window)": s[dwin].sum() / ly_dwin,
         "peak / median day": s.max() / s.median()}
    for m in FC_DAYS.to_period("M").unique():
        r[f"{m.strftime('%b')} share %"] = 100 * s[FC_DAYS.to_period("M") == m].sum() / s.sum()
    rows.append(r)
rep = pd.DataFrame(rows).set_index("scenario")
print(rep.to_string())
print(f"\nLast-year actual on the same series: Diwali window {ly_dwin:,.0f} | all aligned days {LY_ACTUAL.sum():,.0f}")
rep.to_csv(os.path.join(OUT_DIR, "spend_override_summary.csv"))
pd.DataFrame(DAILY, index=FC_DAYS).assign(LAST_YEAR_ACTUAL_ALIGNED=LY_ACTUAL).to_csv(
    os.path.join(OUT_DIR, "spend_override_daily.csv"), index_label="date")

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1.2]})
for ax, (lo, hi) in zip(axes, [(FC_DAYS[0], FC_DAYS[-1]), (D_NOW - pd.Timedelta(days=12), D_NOW + pd.Timedelta(days=12))]):
    m = (FC_DAYS >= lo) & (FC_DAYS <= hi)
    ax.plot(FC_DAYS[m], LY_ACTUAL[m], color="black", lw=2.2, label=f"last-year actual (aligned, {SHIFT.days}d shift)")
    for name, d in DAILY.items():
        ax.plot(FC_DAYS[m], d[m], lw=2.4 if name == BASE_NAME else 1.1,
                ls="--" if name.startswith("original") else "-", label=name)
    ax.axvline(D_NOW, color="grey", ls=":")
axes[0].set_title("Same model, same series, only spend changes (petrol held at the base)")
axes[1].set_title("Zoom around Diwali")
axes[0].set_ylabel("units per day (sample of series)")
axes[0].legend(fontsize=6.5)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "spend_override.png"), dpi=130)
plt.show()
print("Saved to", OUT_DIR)